# Setup env

In [1]:
import pandas as pd
from pathlib import Path
from imdb_utils import utils

In [2]:
#  Constants
data_path = '../../data/bronze/imdb_raw_data/'

# Get the absolute path of where you are right now
src_path = Path.cwd()
project_src = src_path.parents[1]
print(f"Your project source root is: {project_src}")

Your project source root is: c:\Users\davib\Desktop\MSc_DataScience\DataWarehouse\lab_assignment\imbd


# Read raw sources

In [3]:
# Read raw data
title_basics = pd.read_csv(f'{data_path}/title.basics.tsv/data.tsv', sep='\t', na_values='\\N')
title_basics = title_basics[title_basics['titleType'].isin(['tvSeries']) & title_basics['isAdult'] == 0]
name_basics = pd.read_csv(f'{data_path}name.basics.tsv/data.tsv', sep='\t', dtype={'birthYear': 'Int64', 'deathYear': 'Int64'}, na_values='\\N')
title_principals = pd.read_csv(f'{data_path}/title.principals.tsv/data.tsv', sep='\t', na_values='\\N')

C:\Users\davib\AppData\Local\Temp\ipykernel_11964\4145145284.py:2: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  title_basics = pd.read_csv(f'{data_path}/title.basics.tsv/data.tsv', sep='\t', na_values='\\N')


# Dimensional tables

In [5]:
# Create a minimal filter dimension table
dim_filter = title_basics[['tconst', 'titleType', 'isAdult']].copy()
dim_filter = dim_filter[(dim_filter['titleType'] == 'tvSeries') & (dim_filter['isAdult'] == 0)]

# Add nconst by joining with title_principals
dim_filter = dim_filter.merge(
    title_principals[['tconst', 'nconst']],
    on='tconst',
    how='inner'
).drop_duplicates()

# Reorder columns
dim_filter = dim_filter[['nconst', 'tconst', 'titleType', 'isAdult']]

print(f"dim_filter shape: {dim_filter.shape}")
print(f"Sample:\n{dim_filter.head()}")

# Write parquet file
dim_filter.to_parquet(f'{project_src}\\data\\silver\\dim_filter.parquet', compression='snappy', engine='pyarrow')

dim_filter shape: (1133198, 4)
Sample:
      nconst     tconst titleType  isAdult
0  nm0345300  tt0032557  tvSeries      0.0
1  nm0430460  tt0032557  tvSeries      0.0
2  nm0580585  tt0032557  tvSeries      0.0
3  nm0186608  tt0032557  tvSeries      0.0
4  nm0279963  tt0032557  tvSeries      0.0


In [4]:
dim_filter = pd.read_parquet(f'{project_src}\\data\\silver\\dim_filter.parquet')
dim_filter

,nconst,tconst,titleType,isAdult
0,nm0345300,tt0032557,tvSeries,0.0
1,nm0430460,tt0032557,tvSeries,0.0
2,nm0580585,tt0032557,tvSeries,0.0
3,nm0186608,tt0032557,tvSeries,0.0
4,nm0279963,tt0032557,tvSeries,0.0
...,...,...,...,...
1133546,nm8011539,tt9916678,tvSeries,0.0
1133547,nm10749212,tt9916678,tvSeries,0.0
1133548,nm9613196,tt9916678,tvSeries,0.0
1133549,nm10783601,tt9916678,tvSeries,0.0


## dim_profession | bridge_profession_group

In [5]:
# Create dim_profession from unique professions in primaryProfession
# Extract all unique professions from comma-separated primaryProfession values

# Sources:
# name_basics

# 2. Create dim_profession (Vectorized unique extraction)
unique_profs = (
    name_basics['primaryProfession']
    .str.split(',')
    .explode()
    .dropna()
    .unique()
)

dim_profession = pd.DataFrame({'profession_nm': sorted(unique_profs)})
dim_profession['profession_id'] = [f'prf_{i+1}' for i in range(len(dim_profession))]


# 3. Create bridge_profession_group (Vectorized expansion)
# Explode the professions column into individual rows
bridge_profession_group = (
    name_basics[['nconst', 'primaryProfession']]
    .dropna(subset=['primaryProfession'])
    .assign(profession_nm=lambda x: x['primaryProfession'].str.split(','))
    .explode('profession_nm')
)

# 4. Map the IDs using Merge (Replaces the dictionary lookup loop)
bridge_profession_group = (
    bridge_profession_group
    .reset_index()
    .merge(dim_profession.reset_index(), on='profession_nm')
    [['nconst', 'profession_id']]
)

# Create profession_group_id for each unique combination of professions per person
bridge_profession_group['profession_group_id'] = 'p_grp_' + (bridge_profession_group.groupby('nconst').ngroup() + 1).astype(str)

# Load dim_filter and apply inner join to keep only data in dim_filter
bridge_profession_group = bridge_profession_group.merge(
    dim_filter[['nconst']].drop_duplicates(),
    on='nconst',
    how='inner'
)
bridge_profession_group['weighting_factor_prf'] = 1 / bridge_profession_group.groupby('nconst')['profession_id'].transform('count')

# Write parquet files
bridge_profession_group.to_parquet(f'{project_src}\\data\\silver\\bridge_profession_group.parquet', compression='snappy', engine='pyarrow')
dim_profession.to_parquet(f'{project_src}\\data\\silver\\dim_profession.parquet', compression='snappy', engine='pyarrow')


print(f"\nbridge_profession_group shape (after filtering): {bridge_profession_group.shape}")
print(f"Sample bridge data:\n{bridge_profession_group.head(10)}")

del bridge_profession_group
del dim_profession


bridge_profession_group shape (after filtering): (780020, 4)
Sample bridge data:
      nconst profession_id profession_group_id  weighting_factor_prf
0  nm0000001        prf_35             p_grp_1              0.333333
1  nm0000001         prf_1             p_grp_1              0.333333
2  nm0000001        prf_25             p_grp_1              0.333333
3  nm0000002         prf_2             p_grp_2              0.500000
4  nm0000002        prf_35             p_grp_2              0.500000
5  nm0000003         prf_2             p_grp_3              0.333333
6  nm0000003        prf_35             p_grp_3              0.333333
7  nm0000003        prf_26             p_grp_3              0.333333
8  nm0000005        prf_41             p_grp_5              0.333333
9  nm0000005        prf_16             p_grp_5              0.333333


## dim_person

In [6]:
# Create dim_person from name_basics
# Load name_basics data

# Sources:
# name_basics
# bridge_profession_group
# dim_filter

bridge_profession_group = pd.read_parquet(f'{project_src}\\data\\silver\\bridge_profession_group.parquet')
# dim_filter = pd.read_parquet(f'{project_src}\\data\\silver\\dim_filter.parquet')

# Select required columns for dim_person
dim_person = utils.select_cols(name_basics, ['nconst', 'primaryName', 'birthYear', 'deathYear'])

# Join with bridge_profssion_group to get profession_group_id
per_prfession = dim_person.merge(bridge_profession_group, on='nconst', how='left')
# Apply inner join with dim_filter to keep only participants in dim_filter
dim_person = dim_person.merge(
    dim_filter[['nconst']].drop_duplicates(),
    on='nconst',
    how='inner'
)

# Remove duplicates if any and set nconst as index
dim_person = per_prfession.drop_duplicates(subset=['nconst'])
dim_person = dim_person[['nconst','primaryName', 'birthYear', 'deathYear', 'profession_group_id']]

print(f"dim_person shape (after filtering): {dim_person.shape}")
# Write parquet files
dim_person.to_parquet(f'{project_src}\\data\\silver\\dim_person.parquet', compression='snappy', engine='pyarrow')

del dim_person

dim_person shape (after filtering): (10777803, 5)


## dim_roles

In [7]:
# # Create dim_roles from title_principals and title_basics
# # Load title_principals and title_basics data

# Sources:
# title_principals
# title_basics
# dim_filter

# Final dim cols
cols_to_keep = ['role_id', 'nconst', 'tconst', 'titleType', 'category', 'job', 'characters']

# Join title_principals with title_basics to get titleType
dim_roles = title_principals.merge(title_basics[['tconst', 'titleType']], on='tconst', how='left')
# Apply inner join with dim_filter to keep only participants in dim_filter
dim_roles = dim_roles.merge(
    dim_filter[['nconst']].drop_duplicates(),
    on='nconst',
    how='inner'
)
# Create a unique role_id combining nconst, tconst, and ordering
dim_roles['role_id'] = dim_roles['nconst'] + '_' + dim_roles['tconst'] + '_' + dim_roles['ordering'].astype(str)

# Select and reorder columns
dim_roles = utils.select_cols(dim_roles, cols_to_keep)

# Remove duplicates and set role_id as index
dim_roles = dim_roles.drop_duplicates(subset=['role_id'])

# Apply inner join with dim_filter to keep only roles in dim_filter
dim_roles = dim_roles.merge(
    dim_filter[['nconst', 'tconst']].drop_duplicates(),
    on=['nconst', 'tconst'],
    how='inner'
)

print(f"dim_roles shape (after filtering): {dim_roles.shape}")

# Write parquet files
dim_roles.to_parquet(f'{project_src}\\data\\silver\\dim_roles.parquet', compression='snappy', engine='pyarrow')

del dim_roles

MemoryError: Unable to allocate 1.02 GiB for an array with shape (6, 22716739) and data type object

## dim_genre | bridge_genre_group

In [ ]:
# 1. Load Data
# Sources:
# title_basics
# dim_filter

dim_filter = pd.read_parquet(f'{project_src}\\data\\silver\\dim_filter.parquet')

# 2. Create dim_genres (The Dimension Table)
# Extract unique genres using vectorized split and unique()
unique_genres = (
    title_basics['genres']
    .str.split(',')
    .explode()
    .dropna()
    .unique()
)

dim_genres = pd.DataFrame({'genre_nm': sorted(unique_genres)})
dim_genres['genre_id'] = [f'gen_{i+1}' for i in range(len(dim_genres))]

# 3. Create bridge_genres (The Bridge Table)
# Instead of a dictionary lookup, we use explode and then merge with dim_genres
bridge_genres = (
    title_basics[['tconst', 'genres']]
    .dropna(subset=['genres'])
    .assign(genre_nm=lambda x: x['genres'].str.split(','))
    .explode('genre_nm')
)

# Merge to get the genre_id (vectorized replacement for the dictionary lookup)
bridge_genres = (
    bridge_genres.reset_index()
    .merge(dim_genres.reset_index(), on='genre_nm')
    [['tconst', 'genre_id']]
)

# 4. Create genre_group_id
# This stays vectorized as before
bridge_genres['genre_group_id'] = (
    'g_grp_' + 
    (bridge_genres.groupby('tconst').ngroup() + 1).astype(str)
)
# Calculate the weight as 1 divided by the number of genres for that specific title
bridge_genres['weighting_factor_gen'] = 1 / bridge_genres.groupby('tconst')['genre_id'].transform('count')

# Apply inner join with dim_filter to keep only genres for titles in dim_filter
bridge_genres = bridge_genres.merge(
    dim_filter[['tconst']].drop_duplicates(),
    on='tconst',
    how='inner'
)

print(f"dim_genres shape: {dim_genres.shape}")
print(f"bridge_genres shape (after filtering): {bridge_genres.shape}")

# Write parquet files
dim_genres.to_parquet(f'{project_src}\\data\\silver\\dim_genres.parquet', compression='snappy', engine='pyarrow')
bridge_genres.to_parquet(f'{project_src}\\data\\silver\\bridge_genres.parquet', compression='snappy', engine='pyarrow')

del dim_genres
del bridge_genres

dim_genres shape: (28, 2)
bridge_genres shape: (11634945, 4)


## dim_title_basic 

In [ ]:
# Sources:
# title_basics
# bridge_genres
# dim_filter

bridge_genres = pd.read_parquet(f'{project_src}\\data\\silver\\bridge_genres.parquet')
# Select required columns for dim_title_basic
# dim_title_basic = title_basics[['tconst', 'titleType','primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes']].copy()
dim_title_basic = utils.select_cols(title_basics, ['tconst', 'titleType','primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes'])

# Join with bridge_profssion_group to get profession_group_id
title_genre = dim_title_basic.join(bridge_genres.set_index('tconst'), on='tconst', how='left')
# title_genre = title_genre[['tconst', 'titleType','primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes', 'genre_group_id']].copy()
title_genre = utils.select_cols(title_genre, ['tconst', 'titleType','primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes', 'genre_group_id'])
# Remove duplicates if any and set tconst as index
dim_title_basic = title_genre.drop_duplicates(subset=['tconst'])

# Convert to numeric, forcing errors to NaN
dim_title_basic['runtimeMinutes'] = pd.to_numeric(dim_title_basic['runtimeMinutes'], errors='coerce')

# Apply inner join with dim_filter to keep only titles in dim_filter
dim_title_basic = dim_title_basic.merge(
    dim_filter[['tconst']].drop_duplicates(),
    on='tconst',
    how='inner'
)

# dim_title_basic = dim_title_basic[['tconst', 'titleType','primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes', 'genre_group_id']]
print(f"dim_title_basic shape (after filtering): {dim_title_basic.shape}")

#Write parquet files
dim_title_basic.to_parquet(f'{project_src}\\data\\silver\\dim_title_basic.parquet', compression='snappy', engine='pyarrow')

del dim_title_basic

C:\Users\davib\AppData\Local\Temp\ipykernel_20292\2909361170.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dim_title_basic['runtimeMinutes'] = pd.to_numeric(dim_title_basic['runtimeMinutes'], errors='coerce')


dim_title_basic shape: (7695635, 9)


##  bridge_kt_group

In [ ]:
# 1. Load data (Keeping your existing logic)
# Note: Using na_values='\\N' as discussed previously is perfect here

# Sources:
# title_basics
# name_basics 

# 2. Vectorized Expansion (Replacing the two loops)
# We select only the columns we need, split the strings into lists, and "explode" them
bridge_kwn_titles = (
    name_basics[['nconst', 'knownForTitles']]
    .dropna(subset=['knownForTitles'])
    .assign(tconst=lambda x: x['knownForTitles'].str.split(','))
    .explode('tconst')
    [['tconst', 'nconst']]  # Reorder to match your desired output
)

# 3. Vectorized ID Generation
# ngroup() is already vectorized, so we just keep this logic
bridge_kwn_titles['kwn_title_group_id'] = (
    'kwn_t_grp_' + 
    (bridge_kwn_titles.groupby('tconst').ngroup() + 1).astype(str)
)
bridge_kwn_titles['weighting_factor_grp'] = 1 / bridge_kwn_titles.groupby('nconst')['tconst'].transform('count')

# Apply inner join with dim_filter to keep only known titles for participants and titles in dim_filter
bridge_kwn_titles = bridge_kwn_titles.merge(
    dim_filter[['nconst', 'tconst']].drop_duplicates(),
    on=['nconst', 'tconst'],
    how='inner'
)

# 4. Get Unique Titles Set (If still needed for other logic)
known_titles = set(bridge_kwn_titles['tconst'].unique())

print(f"bridge_kwn_titles shape (after filtering): {bridge_kwn_titles.shape}")
bridge_kwn_titles.to_parquet(f'{project_src}\\data\\silver\\bridge_kwn_titles.parquet', compression='snappy', engine='pyarrow')

del bridge_kwn_titles
del dim_filter

bridge_kwn_titles shape: (16851455, 4)


# Factual tables

## participations

In [ ]:
# Read dims
dim_person = pd.read_parquet(f'{project_src}\\data\\silver\\dim_person.parquet')
dim_roles = pd.read_parquet(f'{project_src}\\data\\silver\\dim_roles.parquet')
dim_title_basic = pd.read_parquet(f'{project_src}\\data\\silver\\dim_title_basic.parquet')
bridge_kwn_titles = pd.read_parquet(f'{project_src}\\data\\silver\\bridge_kwn_titles.parquet')

# Convert repetitive strings to categories
dim_roles['category'] = dim_roles['category'].astype('category')
dim_roles['job'] = dim_roles['job'].astype('category')
dim_title_basic['titleType'] = dim_title_basic['titleType'].astype('category')

# If you are using Pandas 2.0+, convert IDs and text to PyArrow strings
# This is drastically more memory efficient than standard 'object' strings
# dim_roles['tconst'] = dim_roles['tconst'].astype('string[pyarrow]')

In [ ]:
import gc

# Merge 1: Titles and Roles
participations_pers = dim_title_basic[['tconst','titleType', 'primaryTitle', 'genre_group_id', 'runtimeMinutes']].merge(
    dim_roles[['role_id','tconst', 'nconst', 'category', 'job', 'characters']],
    on='tconst', 
    how='left'
)

# Optional: If you no longer need the source dataframes in this script, delete them to free RAM
del dim_roles
del dim_title_basic
gc.collect() # Force Python to clear unreferenced memory immediately

# Merge 2: Add Person data
participations_pers = participations_pers.merge(
    dim_person[['nconst', 'primaryName', 'profession_group_id']], 
    on='nconst', 
    how='left'
)
del dim_person
gc.collect()

# Merge 3: Add Bridge
# Ensure bridge_kwn_titles is also pre-filtered to ONLY the columns you need before merging
participations_pers = participations_pers.merge(
    bridge_kwn_titles, 
    on=['tconst', 'nconst'], 
    how='left'
)
del bridge_kwn_titles
gc.collect()

# participations_pers = participations_pers[['nconst', 'primaryName', 'tconst', 'titleType', 'primaryTitle', 'runtimeMinutes', 'genre_group_id', 'category', 'job', 'characters', 'profession_group_id', 'kwn_title_group_id']].copy()
participations_pers = utils.select_cols(participations_pers, ['nconst', 'primaryName', 'tconst', 'titleType', 'primaryTitle', 'runtimeMinutes', 'genre_group_id', 'role_id', 'category', 'job', 'characters', 'profession_group_id', 'kwn_title_group_id'])
participations_pers['participation_count'] = 1

print(f"participations_pers.shape: {participations_pers.shape}")

# #Write parquet file
participations_pers.to_parquet(f'{project_src}\\data\\silver\\participations_pers.parquet', compression='snappy', engine='pyarrow')

# del participations_pers

participations_pers.shape: (44202851, 14)


In [ ]:
participations_pers

,nconst,primaryName,tconst,titleType,primaryTitle,runtimeMinutes,genre_group_id,role_id,category,job,characters,profession_group_id,kwn_title_group_id,participation_count
0,nm0437038,Mika Kanai,tt0112948,tvSeries,The Elven Bride,50.0,g_grp_1,nm0437038_tt0112948_1,actress,None,"[""Noity""]",p_grp_404480,NaN,1
1,nm1013420,Yumi Kuroda,tt0112948,tvSeries,The Elven Bride,50.0,g_grp_1,nm1013420_tt0112948_2,actress,None,"[""Milfa""]",p_grp_1000191,NaN,1
2,nm0559567,Yasunori Matsumoto,tt0112948,tvSeries,The Elven Bride,50.0,g_grp_1,nm0559567_tt0112948_3,actor,None,"[""Kenji""]",p_grp_517304,NaN,1
3,nm0153852,Vanessa Chase,tt0160838,tvSeries,Sex House,NaN,None,nm0153852_tt0160838_10,actress,None,None,p_grp_143269,NaN,1
4,nm0528841,Beverly Lynne,tt0160838,tvSeries,Sex House,NaN,None,nm0528841_tt0160838_1,actress,None,"[""Alexis""]",p_grp_489014,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11667,nm11246633,Aubrey,tt9915708,tvSeries,Atkexotics,NaN,g_grp_1412,nm11246633_tt9915708_5,self,None,"[""Self""]",None,NaN,1
11668,nm12227840,Bailey,tt9915708,tvSeries,Atkexotics,NaN,g_grp_1412,nm12227840_tt9915708_6,self,None,"[""Self""]",p_grp_2328135,NaN,1
11669,nm12230437,Dharma Grace,tt9915708,tvSeries,Atkexotics,NaN,g_grp_1412,nm12230437_tt9915708_7,self,None,"[""Self""]",None,NaN,1
11670,nm12229759,Britney Lee Hunter,tt9915708,tvSeries,Atkexotics,NaN,g_grp_1412,nm12229759_tt9915708_8,self,None,"[""Self""]",p_grp_2329438,kwn_t_grp_1632053,1
